# PrDiMP50 Tracker Training on GOT-10k

This notebook provides a complete training pipeline for the PrDiMP50 tracker with ResNet50 backbone.

**Requirements:**
- GOT-10k dataset
- PyTorch
- torchvision
- PIL
- matplotlib
- numpy

## 1. Setup and Imports

In [ ]:
# Install required packages (uncomment if needed)
# !pip install torch torchvision pillow matplotlib numpy tqdm

import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from datetime import datetime

# Add project root to path
project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root: {project_root}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Import Project Modules

In [ ]:
# Import dataset
from data.got10k_dataset import (
    GOT10kTrainDataset,
    got10k_collate,
    list_sequences
)

# Import model components
from models.backbone import ResNet50Backbone
from models.target_candidate_matching import TargetCandidateMatchingNetwork
from models.bbreg import IoUNet
from models.prdimp import PrDiMPTracker

# Import loss functions
from losses.classification_loss import ClassificationLoss
from losses.iou_loss import IoULoss

print("✓ All modules imported successfully!")

## 3. Configuration

In [ ]:
# ============================================
# CONFIGURATION - MODIFY THESE AS NEEDED
# ============================================

# Dataset Configuration
DATASET_ROOT = "/kaggle/input/got10k"  # <<< CHANGE THIS TO YOUR DATASET PATH
DATASET_SPLIT = "train"

# Model Configuration
TEMPLATE_SIZE = (128, 128)
SEARCH_SIZE = (320, 320)
FEATURE_DIM = 256

# Training Configuration
BATCH_SIZE = 8
NUM_EPOCHS = 50
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 4

# Loss Weights
CLASSIFICATION_WEIGHT = 1.0
IOU_WEIGHT = 1.0

# Device Configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Checkpointing
CHECKPOINT_DIR = "checkpoints"
SAVE_FREQUENCY = 5  # Save every N epochs
LOG_FREQUENCY = 10  # Log every N batches

# Create checkpoint directory
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("=" * 60)
print("CONFIGURATION")
print("=" * 60)
print(f"Dataset Root: {DATASET_ROOT}")
print(f"Split: {DATASET_SPLIT}")
print(f"Template Size: {TEMPLATE_SIZE}")
print(f"Search Size: {SEARCH_SIZE}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Learning Rate: {LEARNING_RATE}")
print(f"Device: {DEVICE}")
print("=" * 60)

## 4. Dataset Preparation

In [ ]:
# Check dataset availability
split_dir = os.path.join(DATASET_ROOT, DATASET_SPLIT)
if not os.path.exists(split_dir):
    raise FileNotFoundError(f"Dataset split directory not found: {split_dir}")

sequences = list_sequences(split_dir)
print(f"✓ Found {len(sequences)} sequences in {DATASET_SPLIT} split")
print(f"Example sequences: {sequences[:5]}")

# Create dataset
train_dataset = GOT10kTrainDataset(
    root_dir=DATASET_ROOT,
    split=DATASET_SPLIT,
    template_size=TEMPLATE_SIZE,
    search_size=SEARCH_SIZE,
    max_search_gap=100,
    template_jitter=0.3,
    search_jitter=1.0,
    rng_seed=42
)

print(f"✓ Dataset created with {len(train_dataset)} indicative samples")

# Create dataloader
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=got10k_collate,
    num_workers=NUM_WORKERS,
    pin_memory=True if torch.cuda.is_available() else False,
    drop_last=True
)

print(f"✓ DataLoader created with batch size {BATCH_SIZE}")
print(f"Approximate batches per epoch: {len(train_loader)}")

## 5. Visualize Sample Batch

In [ ]:
def visualize_batch(batch, num_samples=2):
    """Visualize template and search pairs with ground truth boxes"""
    fig, axes = plt.subplots(num_samples, 2, figsize=(12, 6 * num_samples))
    if num_samples == 1:
        axes = axes.reshape(1, -1)
    
    for i in range(min(num_samples, batch['template'].shape[0])):
        # Template
        tpl_img = batch['template'][i].permute(1, 2, 0).cpu().numpy()
        tpl_gt = batch['tpl_gt'][i].cpu().numpy()
        
        axes[i, 0].imshow(np.clip(tpl_img, 0, 1))
        axes[i, 0].set_title(f"Template - {batch['seq_name'][i][:20]}")
        axes[i, 0].axis('off')
        
        # Draw GT box
        cx, cy, w, h = tpl_gt
        x1, y1 = cx - w/2, cy - h/2
        rect = plt.Rectangle((x1, y1), w, h, fill=False, color='red', linewidth=2)
        axes[i, 0].add_patch(rect)
        
        # Search
        srch_img = batch['search'][i].permute(1, 2, 0).cpu().numpy()
        srch_gt = batch['srch_gt'][i].cpu().numpy()
        
        axes[i, 1].imshow(np.clip(srch_img, 0, 1))
        axes[i, 1].set_title(f"Search - Frame {batch['s_idx'][i]}")
        axes[i, 1].axis('off')
        
        # Draw GT box
        cx, cy, w, h = srch_gt
        x1, y1 = cx - w/2, cy - h/2
        rect = plt.Rectangle((x1, y1), w, h, fill=False, color='green', linewidth=2)
        axes[i, 1].add_patch(rect)
    
    plt.tight_layout()
    plt.show()

# Get sample batch
sample_batch = next(iter(train_loader))
print("Sample Batch Info:")
print(f"  Template shape: {sample_batch['template'].shape}")
print(f"  Search shape: {sample_batch['search'].shape}")
print(f"  Template GT shape: {sample_batch['tpl_gt'].shape}")
print(f"  Search GT shape: {sample_batch['srch_gt'].shape}")

visualize_batch(sample_batch, num_samples=2)

## 6. Model Initialization

In [ ]:
# Initialize backbone
backbone = ResNet50Backbone(pretrained=True)
print("✓ ResNet50 Backbone initialized")

# Initialize Target-Candidate Matching Network
tcm = TargetCandidateMatchingNetwork(feature_dim=FEATURE_DIM)
print("✓ Target-Candidate Matching Network initialized")

# Initialize IoUNet
iou_net = IoUNet(input_dim=FEATURE_DIM)
print("✓ IoUNet initialized")

# Initialize complete PrDiMP tracker
model = PrDiMPTracker(
    backbone=backbone,
    tcm=tcm,
    iou_net=iou_net
)
model = model.to(DEVICE)
print(f"✓ PrDiMP Tracker initialized and moved to {DEVICE}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel Parameters:")
print(f"  Total: {total_params:,}")
print(f"  Trainable: {trainable_params:,}")

## 7. Loss Functions and Optimizer

In [ ]:
# Initialize loss functions
classification_loss_fn = ClassificationLoss()
iou_loss_fn = IoULoss()
print("✓ Loss functions initialized")

# Initialize optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)
print(f"✓ Adam optimizer initialized (lr={LEARNING_RATE}, wd={WEIGHT_DECAY})")

# Learning rate scheduler
scheduler = optim.lr_scheduler.StepLR(
    optimizer,
    step_size=20,
    gamma=0.1
)
print("✓ StepLR scheduler initialized (step_size=20, gamma=0.1)")

## 8. Training Functions

In [ ]:
def train_one_epoch(model, dataloader, optimizer, classification_loss_fn, iou_loss_fn, 
                    device, epoch, log_frequency=10):
    """
    Train for one epoch
    """
    model.train()
    
    running_cls_loss = 0.0
    running_iou_loss = 0.0
    running_total_loss = 0.0
    
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}")
    
    for batch_idx, batch in enumerate(progress_bar):
        # Move data to device
        template = batch['template'].to(device)
        search = batch['search'].to(device)
        tpl_gt = batch['tpl_gt'].to(device)
        srch_gt = batch['srch_gt'].to(device)
        
        # Forward pass
        optimizer.zero_grad()
        
        # Get model outputs
        outputs = model(template, search)
        
        # Compute losses
        cls_loss = classification_loss_fn(
            outputs['classification_scores'],
            srch_gt
        )
        
        iou_loss = iou_loss_fn(
            outputs['bbox_pred'],
            srch_gt
        )
        
        # Total loss
        total_loss = CLASSIFICATION_WEIGHT * cls_loss + IOU_WEIGHT * iou_loss
        
        # Backward pass
        total_loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        # Update running losses
        running_cls_loss += cls_loss.item()
        running_iou_loss += iou_loss.item()
        running_total_loss += total_loss.item()
        
        # Update progress bar
        if (batch_idx + 1) % log_frequency == 0:
            avg_cls = running_cls_loss / (batch_idx + 1)
            avg_iou = running_iou_loss / (batch_idx + 1)
            avg_total = running_total_loss / (batch_idx + 1)
            
            progress_bar.set_postfix({
                'cls_loss': f'{avg_cls:.4f}',
                'iou_loss': f'{avg_iou:.4f}',
                'total': f'{avg_total:.4f}'
            })
    
    # Compute epoch averages
    num_batches = len(dataloader)
    avg_cls_loss = running_cls_loss / num_batches
    avg_iou_loss = running_iou_loss / num_batches
    avg_total_loss = running_total_loss / num_batches
    
    return {
        'cls_loss': avg_cls_loss,
        'iou_loss': avg_iou_loss,
        'total_loss': avg_total_loss
    }


def save_checkpoint(model, optimizer, scheduler, epoch, losses, filepath):
    """
    Save model checkpoint
    """
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'losses': losses
    }
    torch.save(checkpoint, filepath)
    print(f"✓ Checkpoint saved: {filepath}")


def load_checkpoint(model, optimizer, scheduler, filepath):
    """
    Load model checkpoint
    """
    checkpoint = torch.load(filepath, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    epoch = checkpoint['epoch']
    losses = checkpoint['losses']
    print(f"✓ Checkpoint loaded from epoch {epoch}")
    return epoch, losses

print("✓ Training functions defined")

## 9. Training Loop

In [ ]:
# Training history
history = {
    'epochs': [],
    'cls_loss': [],
    'iou_loss': [],
    'total_loss': [],
    'lr': []
}

# Optional: Load from checkpoint
start_epoch = 0
# Uncomment to resume training:
# start_epoch, history = load_checkpoint(model, optimizer, scheduler, 
#                                        'checkpoints/checkpoint_epoch_10.pth')
# start_epoch += 1

print("=" * 60)
print("STARTING TRAINING")
print("=" * 60)
print(f"Starting from epoch: {start_epoch + 1}")
print(f"Total epochs: {NUM_EPOCHS}")
print("=" * 60)

# Training loop
for epoch in range(start_epoch, NUM_EPOCHS):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch + 1}/{NUM_EPOCHS}")
    print(f"Learning Rate: {optimizer.param_groups[0]['lr']:.6f}")
    print(f"{'='*60}")
    
    # Train one epoch
    epoch_losses = train_one_epoch(
        model=model,
        dataloader=train_loader,
        optimizer=optimizer,
        classification_loss_fn=classification_loss_fn,
        iou_loss_fn=iou_loss_fn,
        device=DEVICE,
        epoch=epoch,
        log_frequency=LOG_FREQUENCY
    )
    
    # Update scheduler
    scheduler.step()
    
    # Record history
    history['epochs'].append(epoch + 1)
    history['cls_loss'].append(epoch_losses['cls_loss'])
    history['iou_loss'].append(epoch_losses['iou_loss'])
    history['total_loss'].append(epoch_losses['total_loss'])
    history['lr'].append(optimizer.param_groups[0]['lr'])
    
    # Print epoch summary
    print(f"\nEpoch {epoch + 1} Summary:")
    print(f"  Classification Loss: {epoch_losses['cls_loss']:.4f}")
    print(f"  IoU Loss: {epoch_losses['iou_loss']:.4f}")
    print(f"  Total Loss: {epoch_losses['total_loss']:.4f}")
    
    # Save checkpoint
    if (epoch + 1) % SAVE_FREQUENCY == 0:
        checkpoint_path = os.path.join(
            CHECKPOINT_DIR,
            f"checkpoint_epoch_{epoch + 1}.pth"
        )
        save_checkpoint(model, optimizer, scheduler, epoch, history, checkpoint_path)
    
    # Always save latest checkpoint
    latest_path = os.path.join(CHECKPOINT_DIR, "checkpoint_latest.pth")
    save_checkpoint(model, optimizer, scheduler, epoch, history, latest_path)

print("\n" + "=" * 60)
print("✓ TRAINING COMPLETED!")
print("=" * 60)

# Save final model
final_model_path = os.path.join(CHECKPOINT_DIR, "prdimp50_final.pth")
torch.save(model.state_dict(), final_model_path)
print(f"✓ Final model saved: {final_model_path}")

## 10. Visualize Training History

In [ ]:
def plot_training_history(history):
    """Plot training losses and learning rate"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Classification Loss
    axes[0, 0].plot(history['epochs'], history['cls_loss'], 'b-', linewidth=2)
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Classification Loss')
    axes[0, 0].set_title('Classification Loss over Epochs')
    axes[0, 0].grid(True, alpha=0.3)
    
    # IoU Loss
    axes[0, 1].plot(history['epochs'], history['iou_loss'], 'r-', linewidth=2)
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('IoU Loss')
    axes[0, 1].set_title('IoU Loss over Epochs')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Total Loss
    axes[1, 0].plot(history['epochs'], history['total_loss'], 'g-', linewidth=2)
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Total Loss')
    axes[1, 0].set_title('Total Loss over Epochs')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Learning Rate
    axes[1, 1].plot(history['epochs'], history['lr'], 'm-', linewidth=2)
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Learning Rate')
    axes[1, 1].set_title('Learning Rate Schedule')
    axes[1, 1].set_yscale('log')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(CHECKPOINT_DIR, 'training_history.png'), dpi=150)
    plt.show()
    print(f"✓ Training history plot saved to {CHECKPOINT_DIR}/training_history.png")

plot_training_history(history)

## 11. Model Inference Test

In [ ]:
# Test inference on a sample batch
model.eval()
with torch.no_grad():
    test_batch = next(iter(train_loader))
    template = test_batch['template'].to(DEVICE)
    search = test_batch['search'].to(DEVICE)
    
    outputs = model(template, search)
    
    print("Inference Test:")
    print(f"  Input template shape: {template.shape}")
    print(f"  Input search shape: {search.shape}")
    print(f"  Output keys: {list(outputs.keys())}")
    print(f"  Classification scores shape: {outputs['classification_scores'].shape}")
    print(f"  BBox predictions shape: {outputs['bbox_pred'].shape}")
    print(f"  IoU predictions shape: {outputs['iou_pred'].shape}")
    print("\n✓ Inference test passed!")

## 12. Save Training Summary

In [ ]:
import json

# Save training summary
summary = {
    'dataset_root': DATASET_ROOT,
    'dataset_split': DATASET_SPLIT,
    'num_sequences': len(sequences),
    'template_size': TEMPLATE_SIZE,
    'search_size': SEARCH_SIZE,
    'batch_size': BATCH_SIZE,
    'num_epochs': NUM_EPOCHS,
    'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'classification_weight': CLASSIFICATION_WEIGHT,
    'iou_weight': IOU_WEIGHT,
    'device': str(DEVICE),
    'total_parameters': total_params,
    'trainable_parameters': trainable_params,
    'final_losses': {
        'cls_loss': history['cls_loss'][-1],
        'iou_loss': history['iou_loss'][-1],
        'total_loss': history['total_loss'][-1]
    },
    'training_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
}

summary_path = os.path.join(CHECKPOINT_DIR, 'training_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=4)

print("Training Summary:")
print(json.dumps(summary, indent=2))
print(f"\n✓ Summary saved to {summary_path}")

## Done!

The PrDiMP50 tracker has been trained successfully. The following files have been saved:

- Model checkpoints in `checkpoints/` directory
- Final model: `checkpoints/prdimp50_final.pth`
- Training history plot: `checkpoints/training_history.png`
- Training summary: `checkpoints/training_summary.json`

To use the trained model for tracking, load it with:
```python
model.load_state_dict(torch.load('checkpoints/prdimp50_final.pth'))
model.eval()
```